In [1]:
from Lists.abbreviations import INTERNET_SLANG
from Lists.emoticons import EMOTICONS
from Lists.emojis import EMO_UNICODE
import pandas as pd
import contractions
import unicodedata
import re

In [3]:
df = pd.read_csv('./cyberbullying/twitter_ethnicity.csv')
df=df.dropna(how='any')
df.head()

,text,label,type
0,@AAlwuhaib1977 Muslim mob violence against Hin...,4,ethnicity
1,@jncatron @isra_jourisra @AMPalestine Islamoph...,4,ethnicity
2,@HuffPostRelig Islam invaded and conquered 2/3...,4,ethnicity
3,@semzyxx Do you approve of your pedophile prop...,4,ethnicity
4,@watan71969 @geeky_zekey Problem with vile Mus...,4,ethnicity


In [4]:
UNICODE_EMO = {v: k for k, v in EMO_UNICODE.items()}


def convert_emoticons(text):
    for emot in EMOTICONS:
        text = re.sub(u'(' + emot + ')', "  ".join(EMOTICONS[emot].replace(",", "").split()), text)
    return text


def convert_emojis(text):
    for emot in UNICODE_EMO:
        text = re.sub(r'(' + emot + ')', "  ".join(UNICODE_EMO[emot].replace(",", "").replace(":", "").split()), text)
    return text


def convert_abbreviations(text):
    for abbr in INTERNET_SLANG:
        pattern = re.compile(r'\b' + re.escape(abbr) + r'\b', re.IGNORECASE)
        replacement = " ".join(INTERNET_SLANG[abbr].replace(",", "").split())
        text = pattern.sub(replacement, text)
    return text


def remove_unknown_emojis(text):
    emoji_pattern = re.compile("["
                               u"\U0001F600-\U0001F64F"  # emoticons
                               u"\U0001F300-\U0001F5FF"  # symbols & pictographs
                               u"\U0001F680-\U0001F6FF"  # transport & map symbols
                               u"\U0001F1E0-\U0001F1FF"  # flags (iOS)
                               u"\U00002702-\U000027B0"  # miscellaneous symbols
                               u"\U0001F900-\U0001F9FF"  # supplemental symbols
                               u"\U0001F200-\U0001F251"  # enclosed characters
                               u"\U0001F004-\U0001F0CF"  # playing cards
                               u"\U00002B50"  # star symbol
                               "]+", flags=re.UNICODE)

    emojis_in_text = emoji_pattern.findall(text)

    for emoji_char in emojis_in_text:
        if emoji_char not in UNICODE_EMO:
            text = text.replace(emoji_char, '')
    return text


def remove_accented_chars(text):
    text = contractions.fix(text)
    return unicodedata.normalize('NFKD', text).encode('ascii', 'ignore').decode('utf-8', 'ignore')

In [5]:
def convert_emojis_and_slang(text):
    text = convert_emoticons(text)
    text = convert_emojis(text)
    text = remove_unknown_emojis(text)
    text = convert_abbreviations(text)
    text=remove_accented_chars(text)

    return text

In [6]:
from tqdm.notebook import tqdm

In [7]:
def preprocess_text(df_input):
    tqdm.pandas() 

    df_comments = df_input.copy()
    df_comments['text'] = df_comments['text'].str.replace(r"http[s]?://\S+", "URL", regex=True) # remove URLs
    df_comments['text'] = df_comments['text'].str.replace(r'@[A-Za-z0-9_.]+', 'USER', regex=True)  # remove user mentions
    df_comments['text'] = df_comments['text'].str.replace(r'#+(\S+)', r'\1', regex=True)  # remove hashtags
    df_comments['text'] = df_comments['text'].str.replace(r'\s+', ' ', regex=True)  # remove extra spaces
    df_comments['text'] = df_comments['text'].str.replace(r'\d+', '', regex=True)  # remove digits

    df_comments['text'] = df_comments['text'].progress_apply(lambda x: convert_emojis_and_slang(x))

    df_comments = df_comments.dropna(how='any')
    df_comments = df_comments.drop_duplicates()

    return df_comments


In [8]:
df = preprocess_text(df)

  0%|          | 0/1970 [00:00<?, ?it/s]

In [10]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1965 entries, 0 to 1969
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   text    1965 non-null   object
 1   label   1965 non-null   int64 
 2   type    1965 non-null   object
dtypes: int64(1), object(2)
memory usage: 61.4+ KB


In [14]:
df2 = pd.read_csv('./cyberbullying_tweets_transformers.csv')
rows_to_remove = df2[(df2['type'] == 'ethnicity') & (df2['label'] == 4)].iloc[:1965].index
df2 = df2.drop(index=rows_to_remove)
df2 = pd.concat([df2, df], ignore_index=True)
print(df2[df2['type'] == 'ethnicity']['label'].value_counts())

label
4    7961
Name: count, dtype: int64


In [ ]:
df2 = preprocess_text(df2)

  0%|          | 0/39739 [00:00<?, ?it/s]

In [16]:
df2.to_csv("cyberbullying_tweets_transformers_updated.csv", index=False)